In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable


In [0]:
SILVER_PATH = "s3a://nyc-lakehouse/silver/nyc_taxi/"
GOLD_PATH = "s3a://nyc-lakehouse/gold/nyc_taxi/"

In [0]:
# Read data from Silver layer
silver_df = spark.read.format('delta').table('nyc_silver_yellow')

print(f"Silver layer record count: {silver_df.count()}")

Silver layer record count: 110820241


In [0]:
def write_gold_table(
    spark,
    df,
    table_name,
    path,
    partition_cols=None,
    merge_keys=None
):

    if not DeltaTable.isDeltaTable(spark, path):

        writer = df.write.format("delta")

        if partition_cols:
            writer = writer.partitionBy(*partition_cols)

        writer.mode("overwrite").save(path)

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_name}
        USING DELTA
        LOCATION '{path}'
        """)

        print(f"Created {table_name}")
        return

    delta_tbl = DeltaTable.forPath(spark, path)

    if merge_keys:
        cond = " AND ".join(
            [f"t.{k}=s.{k}" for k in merge_keys]
        )

        (
            delta_tbl.alias("t")
            .merge(df.alias("s"), cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .save(path)
        )

    print(f"Updated {table_name}")


In [0]:

def create_daily_statistics(df):
    """
    Create daily aggregated statistics
    """

    daily_stats = df.groupBy("pickup_date").agg(
        count("*").alias("total_trips"),
        sum("passenger_count").alias("total_passengers"),
        avg("passenger_count").alias("avg_passengers_per_trip"),
        sum("trip_distance").alias("total_distance"),
        avg("trip_distance").alias("avg_trip_distance"),
        sum("trip_duration_minutes").alias("total_trip_duration_minutes"),
        avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
        sum("fare_amount").alias("total_fare"),
        avg("fare_amount").alias("avg_fare"),
        sum("tip_amount").alias("total_tips"),
        avg("tip_amount").alias("avg_tip"),
        avg("tip_percentage").alias("avg_tip_percentage"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_revenue_per_trip"),
        avg("average_speed_mph").alias("avg_speed_mph")

    ).orderBy("pickup_date")
    
    # Add calculated KPIs
    daily_stats = daily_stats.withColumn(
        "revenue_per_mile",
        col("total_revenue") / col("total_distance")
    ).withColumn(
        "revenue_per_minute",
        col("total_revenue") / col("total_trip_duration_minutes")
    ).withColumn(
        "processing_timestamp",
        current_timestamp()
    )

    w = Window.orderBy("pickup_date")

    daily_stats = daily_stats.withColumn("prev_day_trips", lag("total_trips").over(w)
    ).withColumn(
            "trip_growth_pct",
            ((col("total_trips")-col("prev_day_trips")) /
             col("prev_day_trips"))*100
    ).withColumn(
            "rolling_7d_avg_trips",
            avg("total_trips").over(w.rowsBetween(-6,0))
    ).withColumn("processing_ts", current_timestamp())
    
    
    return daily_stats



In [0]:

def create_hourly_patterns(df):
    """
    Analyze trip patterns by hour of day
    """
    df = df.repartition("pickup_hour")
                        
    hourly_patterns = df.groupBy("pickup_hour").agg(
        count("*").alias("total_trips"),
        avg("trip_distance").alias("avg_trip_distance"),
        avg("trip_duration_minutes").alias("avg_trip_duration"),
        avg("fare_amount").alias("avg_fare"),
        avg("tip_percentage").alias("avg_tip_percentage"),
        avg("passenger_count").alias("avg_passengers"),
        sum("total_amount").alias("total_revenue")
    ).orderBy("pickup_hour")
    
    # Add time of day classification
    hourly_patterns = hourly_patterns.withColumn(
        "time_of_day",
        when((col("pickup_hour") >= 6) & (col("pickup_hour") < 12), lit("Morning"))
        .when((col("pickup_hour") >= 12) & (col("pickup_hour") < 17), lit("Afternoon"))
        .when((col("pickup_hour") >= 17) & (col("pickup_hour") < 21), lit("Evening"))
        .otherwise(lit("Night"))
    ).withColumn(
        "processing_timestamp",
        current_timestamp()
    )
    
    return hourly_patterns



In [0]:

def create_location_analytics(df):
    """
    Analyze pickup and dropoff location performance
    """

    df = df.repartition("PULocationID", "DOLocationID")
    
    pickup_stats = df.groupBy("PULocationID").agg(
        count("*").alias("total_pickups"),
        avg("trip_distance").alias("avg_trip_distance"),
        avg("fare_amount").alias("avg_fare"),
        sum("total_amount").alias("total_revenue")
    )#.withColumnRenamed("PULocationID", "location_id")
    
    dropoff_stats = df.groupBy("DOLocationID").agg(
        count("*").alias("total_dropoffs")
    )#.withColumnRenamed("DOLocationID", "location_id")
    
    # Combine pickup and dropoff stats
    location_analytics = pickup_stats.join(
        dropoff_stats,
        on=pickup_stats["PULocationID"] == dropoff_stats["DOLocationID"],
        how="outer"
    ).fillna(0)
    
    # Calculate popularity score
    location_analytics = location_analytics.withColumn(
        "popularity_score",
        col("total_pickups") + col("total_dropoffs")
    ).withColumn(
        "processing_timestamp",
        current_timestamp()
    ).orderBy(desc("popularity_score"))

    total_rev = location_analytics.agg(sum("total_revenue")).collect()[0][0]

    location_analytics = location_analytics.withColumn(
        "revenue_share_pct",
        (col("total_revenue")/lit(total_rev))*100
    ).orderBy(desc("revenue_share_pct"))
    
    return location_analytics



In [0]:

def create_payment_analysis(df):
    """
    Analyze payment types and tipping behavior
    """
    df = df.repartition("payment_type", "is_weekend")

    payment_analysis = df.groupBy("payment_type", "is_weekend").agg(
        count("*").alias("total_trips"),
        avg("fare_amount").alias("avg_fare"),
        avg("tip_amount").alias("avg_tip"),
        avg("tip_percentage").alias("avg_tip_percentage"),
        sum("total_amount").alias("total_revenue"),
        percentile_approx("tip_percentage", 0.5).alias("median_tip_percentage")
    ).orderBy("payment_type", "is_weekend")
    
    # Add payment type description
    payment_analysis = payment_analysis.withColumn(
        "payment_method",
        when(col("payment_type") == 1, lit("Credit Card"))
        .when(col("payment_type") == 2, lit("Cash"))
        .when(col("payment_type") == 3, lit("No Charge"))
        .when(col("payment_type") == 4, lit("Dispute"))
        .otherwise(lit("Unknown"))
    ).withColumn(
        "day_type",
        when(col("is_weekend") == True, lit("Weekend"))
        .otherwise(lit("Weekday"))
    ).withColumn(
        "processing_timestamp",
        current_timestamp()
    )
    
    return payment_analysis



In [0]:

def create_business_summary(df):
    """
    Create overall business summary metrics
    """
    # Overall metrics
    overall_metrics = df.agg(
        count("*").alias("total_trips"),
        countDistinct("pickup_date").alias("total_days"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_revenue_per_trip"),
        sum("trip_distance").alias("total_miles"),
        avg("trip_distance").alias("avg_trip_distance"),
        avg("passenger_count").alias("avg_passengers"),
        avg("tip_percentage").alias("avg_tip_percentage")
    )
    
    # Add calculated metrics
    summary = overall_metrics.withColumn(
        "avg_daily_trips",
        col("total_trips") / col("total_days")
    ).withColumn(
        "avg_daily_revenue",
        col("total_revenue") / col("total_days")
    ).withColumn(
        "revenue_per_mile",
        col("total_revenue") / col("total_miles")
    ).withColumn(
        "processing_timestamp",
        current_timestamp()
    ).withColumn(
        "report_period_start",
        lit(df.agg(min("pickup_date")).collect()[0][0])
    ).withColumn(
        "report_period_end",
        lit(df.agg(max("pickup_date")).collect()[0][0])
    )
    
    return summary



In [0]:
daily_df = create_daily_statistics(silver_df)

write_gold_table(
    spark,
    daily_df,
    "nyc_gold_daily_stats",
    GOLD_PATH+"daily_stats",
    partition_cols=["pickup_date"],
    merge_keys=["pickup_date"]
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Updated nyc_gold_daily_stats


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
hourly_df = create_hourly_patterns(silver_df)

write_gold_table(
    spark,
    hourly_df,
    "nyc_gold_hourly_stats",
    GOLD_PATH + "hourly_stats",
    partition_cols=None,
    merge_keys=["pickup_hour"]
)


Updated nyc_gold_hourly_stats


In [0]:
location_df = create_location_analytics(silver_df)

write_gold_table(
    spark,
    location_df,
    "nyc_gold_location_stats",
    GOLD_PATH+"location_stats",
    partition_cols=None,
    merge_keys=["PULocationID"]
)


Updated nyc_gold_location_stats


In [0]:
payment_df = create_payment_analysis(silver_df)

write_gold_table(
    spark,
    payment_df,
    "nyc_gold_payment_stats",
    GOLD_PATH+"payment_stats",
    partition_cols=None,
    merge_keys=["payment_type", "is_weekend"]
)

Updated nyc_gold_payment_stats


In [0]:
business_df = create_business_summary(silver_df)

write_gold_table(
    spark,
    business_df,
    "nyc_gold_business_stats",
    GOLD_PATH+"business_stats",
    partition_cols=None,
    merge_keys=None
)

Updated nyc_gold_business_stats
